# Clasificación de Iris con Random Forest

Este notebook entrena un `RandomForestClassifier` sobre `Iris.csv`, evalúa su desempeño y registra el resultado en MLflow y Unity Catalog. La lógica común vive en `tools/src/iris_mlflow_utils`; aquí solo se mantiene la configuración y la elección del algoritmo.

## 1. Dependencias

En Databricks se instala el paquete local en modo editable para que las funciones de `tools` estén disponibles. La instalación puede reiniciar Python; por eso esta celda debe ejecutarse antes de los imports.

In [0]:
%pip install ./tools
try:
    dbutils.library.restartPython()
except NameError:
    pass

## 2. Imports y configuración del modelo

La configuración compartida lee variables de entorno o widgets de Databricks y aplica valores reproducibles. El único detalle específico de este notebook es el estimador Random Forest y sus hiperparámetros.

In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier

from iris_mlflow_utils import build_config, evaluate_train_test, load_dataset, log_training_run, split_dataset

config = build_config(
    model_slug="random_forest",
    registered_model_name="workspace.default.iris_random_forest",
    run_name="random-forest-iris",
)
model_params = {
    "n_estimators": 100,
    "max_depth": 4,
    "random_state": config.random_state,
    "n_jobs": -1,
}
print(f"Dataset: {config.dataset_path}")
print(f"Experimento: {config.experiment_name}")
print(f"Modelo registrado: {config.registered_model_name}")

## 3. Carga y validación de datos

`load_dataset` verifica la ruta, el objetivo, los valores nulos y los tipos numéricos. Excluye `Id` de las features, codifica `Species` de forma reproducible y conserva el mapping de clases para registrarlo como artefacto.

In [0]:
dataset = load_dataset(config.dataset_path)
print(f"Registros: {len(dataset.dataframe)}")
print(f"Features: {list(dataset.feature_columns)}")
print(f"Clases: {list(dataset.classes)}")
display(dataset.dataframe.head())

## 4. División train/test

El split estratificado conserva la proporción de clases en ambos subconjuntos. `RANDOM_STATE` permite repetir exactamente la partición cuando las versiones de datos y dependencias se mantienen constantes.

In [0]:
split = split_dataset(dataset, config.test_size, config.random_state)
print(f"Train: {len(split.x_train)} filas | Test: {len(split.x_test)} filas")

## 5. Entrenamiento y evaluación

Random Forest combina árboles de decisión y limita la profundidad para mantener un baseline sencillo. Las funciones comunes calculan las mismas métricas para train y test, evitando diferencias entre notebooks.

In [0]:
model = RandomForestClassifier(**model_params)
model.fit(split.x_train, split.y_train)
evaluations = evaluate_train_test(model, split, len(dataset.classes))
metrics = {
    f"{partition}_{name}": value
    for partition, result in evaluations.items()
    for name, value in result.metrics.items()
}
print(metrics)

## 6. Registro en MLflow y Unity Catalog

`log_training_run` crea o reutiliza el experimento, registra parámetros, tags, métricas, reportes, matriz de confusión, mapping de clases, signature e input example. También registra el modelo con el nombre de tres niveles configurado para Unity Catalog.

In [0]:
run_result = log_training_run(
    model=model,
    model_type="RandomForest",
    model_params=model_params,
    config=config,
    split=split,
    evaluations=evaluations,
    feature_columns=dataset.feature_columns,
    classes=dataset.classes,
)
print(f"Run ID: {run_result.run_id}")
print(f"Model URI: {run_result.model_uri}")
print(f"Modelo registrado: {run_result.registered_model_name}")
print(f"Versión registrada: {run_result.registered_model_version}")
print(f"Métrica principal: {run_result.metrics[config.primary_metric]:.4f}")

## 7. Verificación del modelo

Se carga el artefacto desde la URI del run y se ejecuta una predicción sobre filas de prueba. Esta comprobación confirma que el modelo registrado conserva la firma de entrada esperada.

In [0]:
loaded_model = mlflow.sklearn.load_model(run_result.model_uri)
predictions = loaded_model.predict(split.x_test.head(3))
print(f"Predicciones codificadas: {predictions.tolist()}")
print(f"Clases: {list(dataset.classes)}")